# 03 — Fourier and spectral analysis

The FNO's inductive bias should be motivated by the data, not by architecture fashion.

For a 2-D field \(u(x,y)\),

\[
\widehat u(k_x,k_y)=\mathcal F\{u\},\qquad P(k_x,k_y)=|\widehat u(k_x,k_y)|^2.
\]

We ask how SST variance is distributed over spatial frequencies and how much structure is retained by low-mode approximations.

**Important:** land masks create sharp discontinuities and therefore artificial high-frequency Fourier energy. The diagnostics below use a common mask and Hann taper. They are useful for relative model comparisons, not as uncontaminated physical ocean spectra.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from oisst_fno.data import open_oisst

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
sst = open_oisst(path)["sst"]
field = sst.isel(time=len(sst.time) // 2).values.astype(np.float32)
mask = np.isfinite(field)
fill_value = float(np.nanmean(field))
filled = np.where(mask, field, fill_value)

valid_mean = float(np.mean(filled[mask]))
centered = np.where(mask, filled - valid_mean, 0.0)
taper = np.outer(np.hanning(centered.shape[0]), np.hanning(centered.shape[1]))
analysis_field = centered * mask * taper

In [ ]:
spectrum = np.fft.fftshift(np.fft.fft2(analysis_field, norm="ortho"))
power = np.abs(spectrum) ** 2

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(np.log10(power + 1e-8), origin="lower", aspect="auto")
ax.set_title("Log 2-D spatial power spectrum\n(common mask + Hann taper)")
fig.colorbar(im, ax=ax, label="log10 power")
plt.show()

In [ ]:
def low_pass_reconstruct(field_2d: np.ndarray, modes_y: int, modes_x: int) -> np.ndarray:
    """Illustrative low-mode reconstruction on the already tapered analysis field."""
    ft = np.fft.rfft2(field_2d, norm="ortho")
    kept = np.zeros_like(ft)
    kept[:modes_y, :modes_x] = ft[:modes_y, :modes_x]
    kept[-modes_y:, :modes_x] = ft[-modes_y:, :modes_x]
    return np.fft.irfft2(kept, s=field_2d.shape, norm="ortho").real

for modes in (4, 8, 16, 24):
    reconstructed = low_pass_reconstruct(analysis_field, modes, modes)
    relative_error = np.linalg.norm(analysis_field - reconstructed) / np.linalg.norm(analysis_field)
    print(f"modes={modes:2d}: tapered-field relative reconstruction error={relative_error:.4f}")

This is only architecture motivation. It does **not** show that FNO will forecast well.

The decisive spectral question comes after prediction:

\[
E_m(k)=\left|\mathcal F(\hat u_m-u)\right|^2,
\]

and we compare the FNO's error energy against persistence in predefined radial-frequency bands.

If aggregate RMSE improves while high-frequency error gets worse, the model may simply be purchasing RMSE through smoothing.

### Stronger follow-up

For publication-quality spectral interpretation, repeat these diagnostics on one or more **fully oceanic rectangular subdomains**. That reduces mask-induced leakage and makes the frequency interpretation cleaner.